# Step 3 — Time split + leak-free features (IBM AML HI-Small)

**Runs on Kaggle.** Add the dataset *IBM Transactions for Anti Money Laundering (AML)* as input → **Run All** (~5–10 min).

1. Load the 5.08M transactions; attach laundering-pattern labels.
2. Time split on the **dense period** (the sparse tail days Sep 11–18 are kept separate, as found in Step 2).
3. Build 39 features using **only strictly-earlier transactions** (code = `src/aml/features.py`, unit-tested).
4. Spot-check 300 real rows against a brute-force recomputation.
5. Export small files for the laptop.

In [ ]:
import os, glob, json, time
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
NEGATIVE_RATE = 0.08            # keep 8% of normal txns in sampled train/val (all laundering kept)
SPLIT = (0.60, 0.20, 0.20)      # train / val / test share of dense-period transactions

search_root = os.environ.get("AML_DATA_DIR", "/kaggle/input")
trans_files = glob.glob(f"{search_root}/**/HI-Small_Trans.csv", recursive=True)
pattern_files = glob.glob(f"{search_root}/**/HI-Small_Patterns.txt", recursive=True)
assert trans_files, f"HI-Small_Trans.csv not found under {search_root}. Add the dataset as input."
OUT = Path(os.environ.get("AML_OUT_DIR", "/kaggle/working")); OUT.mkdir(parents=True, exist_ok=True)
info = {"pandas_version": pd.__version__}
print("pandas", pd.__version__, "| data:", trans_files[0])

## Feature code (identical copy of `src/aml/features.py`)

In [ ]:
# ---- copied verbatim from src/aml/features.py (tested by tests/test_features.py) ----
"""Leak-free transaction features.

Every feature for a transaction at time t uses ONLY transactions with timestamp < t
(strictly earlier). Transactions in the same minute are treated as simultaneous and
never see each other. This is what makes the time-based evaluation honest.

Uses only numpy + pandas, so the same file runs on Kaggle and on the laptop.

Input: a DataFrame with the standard columns from configs/schema.yaml
(timestamp, from_bank, from_account, to_bank, to_account, amount_received,
receiving_currency, amount_paid, payment_currency, payment_format, [label]).
"""

from __future__ import annotations

import math

import numpy as np
import pandas as pd

HOUR_NS = 3_600 * 10**9
WINDOWS = {"1h": 1 * HOUR_NS, "24h": 24 * HOUR_NS, "7d": 7 * 24 * HOUR_NS}
ALL_HISTORY = 10**18  # effectively infinite window (≈31 years in ns)

CATEGORICAL = ["payment_format", "payment_currency"]


# --------------------------------------------------------------------------- #
# Core algorithm: sliding-window statistics with two pointers + a hash map
# --------------------------------------------------------------------------- #
def window_stats(q_key, q_time, e_key, e_time, e_cpty, e_amount, windows: dict[str, int]):
    """For each query (key k, time t) and each window W, aggregate the events with
    the same key and event time in [t - W, t):
        cnt, distinct counterparties, sum and sum of squares of amount.
    Also returns `last_t`: time of the latest event with time < t (or -1).

    One pass over sorted queries/events, O(n) amortised. Results are returned
    in the ORIGINAL query order.
    """
    q_key = np.asarray(q_key, dtype=np.int64); q_time = np.asarray(q_time, dtype=np.int64)
    e_key = np.asarray(e_key, dtype=np.int64); e_time = np.asarray(e_time, dtype=np.int64)
    e_cpty = np.asarray(e_cpty, dtype=np.int64); e_amount = np.asarray(e_amount, dtype=np.float64)

    qo = np.lexsort((q_time, q_key)); eo = np.lexsort((e_time, e_key))
    QK, QT = q_key[qo].tolist(), q_time[qo].tolist()
    EK, ET = e_key[eo].tolist(), e_time[eo].tolist()
    EC, EA = e_cpty[eo].tolist(), e_amount[eo].tolist()
    nq, ne = len(QK), len(EK)
    names = list(windows); widths = [windows[w] for w in names]; nw = len(names)

    out_cnt = [[0] * nq for _ in range(nw)]; out_dst = [[0] * nq for _ in range(nw)]
    out_sum = [[0.0] * nq for _ in range(nw)]; out_ss = [[0.0] * nq for _ in range(nw)]
    out_last = [-1] * nq

    add = 0
    rem = [0] * nw
    cnt_map = [dict() for _ in range(nw)]
    n_in = [0] * nw; s = [0.0] * nw; ss = [0.0] * nw
    cur_key = None; last_t = -1

    for qi in range(nq):
        k = QK[qi]; t = QT[qi]
        if k != cur_key:  # new account: reset all state
            cur_key = k; last_t = -1
            while add < ne and EK[add] < k:
                add += 1
            for w in range(nw):
                rem[w] = add; cnt_map[w].clear(); n_in[w] = 0; s[w] = 0.0; ss[w] = 0.0
        # add events of this key that happened strictly before t
        while add < ne and EK[add] == k and ET[add] < t:
            c = EC[add]; a = EA[add]; last_t = ET[add]
            for w in range(nw):
                m = cnt_map[w]; m[c] = m.get(c, 0) + 1
                n_in[w] += 1; s[w] += a; ss[w] += a * a
            add += 1
        # drop events older than t - W (per window)
        for w in range(nw):
            lo = t - widths[w]; r = rem[w]; m = cnt_map[w]
            while r < add and ET[r] < lo:
                c = EC[r]; a = EA[r]; v = m[c] - 1
                if v: m[c] = v
                else: del m[c]
                n_in[w] -= 1; s[w] -= a; ss[w] -= a * a
                r += 1
            rem[w] = r
            out_cnt[w][qi] = n_in[w]; out_dst[w][qi] = len(m)
            out_sum[w][qi] = s[w]; out_ss[w][qi] = ss[w]
        out_last[qi] = last_t

    inv = np.empty(nq, dtype=np.int64); inv[qo] = np.arange(nq)
    res = {}
    for w, name in enumerate(names):
        res[f"cnt_{name}"] = np.asarray(out_cnt[w])[inv]
        res[f"distinct_{name}"] = np.asarray(out_dst[w])[inv]
        res[f"sum_{name}"] = np.asarray(out_sum[w])[inv]
        res[f"sumsq_{name}"] = np.asarray(out_ss[w])[inv]
    res["last_t"] = np.asarray(out_last)[inv]
    return res


# --------------------------------------------------------------------------- #
# Exchange rates (estimated from the data itself)
# --------------------------------------------------------------------------- #
def estimate_usd_rates(df: pd.DataFrame) -> dict[str, float]:
    """USD per unit of each currency, estimated from cross-currency transactions
    (median of amount_received / amount_paid). Pass TRAIN-period rows only."""
    rates = {"US Dollar": 1.0}
    pc = df["payment_currency"].astype(str); rc = df["receiving_currency"].astype(str)
    ok = (df["amount_paid"] > 0) & (df["amount_received"] > 0)
    to_usd = df[ok & (rc == "US Dollar") & (pc != "US Dollar")]
    for cur, g in to_usd.groupby(pc[to_usd.index]):
        rates[cur] = float((g["amount_received"] / g["amount_paid"]).median())
    from_usd = df[ok & (pc == "US Dollar") & (rc != "US Dollar")]
    for cur, g in from_usd.groupby(rc[from_usd.index]):
        if cur not in rates:
            rates[cur] = float(1.0 / (g["amount_received"] / g["amount_paid"]).median())
    # Chain through other currencies (e.g. Yen -> Euro -> USD) for any still missing.
    cross = df[ok & (pc != rc)]
    pair_ratio = (cross["amount_received"] / cross["amount_paid"]).groupby(
        [pc[cross.index], rc[cross.index]]).median()            # units of rc per unit of pc
    changed = True
    while changed:
        changed = False
        for (p, r), ratio in pair_ratio.items():
            if p not in rates and r in rates:      # 1 p = ratio r  -> usd(p) = ratio * usd(r)
                rates[p] = float(ratio * rates[r]); changed = True
            elif r not in rates and p in rates:    # 1 r = p / ratio
                rates[r] = float(rates[p] / ratio); changed = True
    return rates


# --------------------------------------------------------------------------- #
# Feature builder
# --------------------------------------------------------------------------- #
def _std(cnt, s, ss):
    with np.errstate(invalid="ignore", divide="ignore"):
        mean = np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan)
        var = np.where(cnt > 1, ss / np.maximum(cnt, 1) - mean**2, np.nan)
    return mean, np.sqrt(np.clip(var, 0, None))


def build_features(df: pd.DataFrame, usd_rates: dict[str, float]) -> pd.DataFrame:
    """Return a DataFrame of features, aligned row-by-row with `df`."""
    df = df.reset_index(drop=True)
    # Force nanosecond unit: pandas 3 may store datetimes in s/us, which would
    # silently make every window 1000x+ too wide.
    t = df["timestamp"].astype("datetime64[ns]").astype("int64").to_numpy()

    # Accounts are identified by (bank, account) to avoid ID collisions across banks.
    src = df["from_bank"].astype(str) + "_" + df["from_account"].astype(str)
    dst = df["to_bank"].astype(str) + "_" + df["to_account"].astype(str)
    codes, uniques = pd.factorize(pd.concat([src, dst], ignore_index=True))
    n = len(df); n_acc = len(uniques)
    s_code = codes[:n].astype(np.int64); r_code = codes[n:].astype(np.int64)

    rate = df["payment_currency"].astype(str).map(usd_rates).astype(float).fillna(1.0).to_numpy()
    amt = df["amount_paid"].to_numpy(dtype=np.float64) * rate

    f = pd.DataFrame(index=df.index)

    # 1) Sender's outgoing history (velocity, fan-out, amount behaviour)
    so = window_stats(s_code, t, s_code, t, r_code, amt, WINDOWS)
    for w in WINDOWS:
        f[f"s_out_cnt_{w}"] = so[f"cnt_{w}"]
        f[f"s_out_uniq_{w}"] = so[f"distinct_{w}"]
    f["s_out_usd_24h"] = so["sum_24h"]; f["s_out_usd_7d"] = so["sum_7d"]
    mean7, std7 = _std(so["cnt_7d"], so["sum_7d"], so["sumsq_7d"])
    with np.errstate(invalid="ignore", divide="ignore"):
        z = (amt - mean7) / std7
    f["s_amt_z_7d"] = np.where(np.isfinite(z), z, 0.0)
    f["s_amt_ratio_7d"] = np.where(so["cnt_7d"] > 0, amt / np.maximum(mean7, 1e-9), 1.0)
    f["s_hours_since_prev_out"] = np.where(so["last_t"] >= 0, (t - so["last_t"]) / HOUR_NS, -1.0)

    # 2) Receiver's incoming history (fan-in)
    ri = window_stats(r_code, t, r_code, t, s_code, amt, WINDOWS)
    for w in WINDOWS:
        f[f"r_in_cnt_{w}"] = ri[f"cnt_{w}"]
        f[f"r_in_uniq_{w}"] = ri[f"distinct_{w}"]
    f["r_in_usd_24h"] = ri["sum_24h"]

    # 3) Cross-role: did the sender just RECEIVE money? does the receiver SEND onward?
    #    (pass-through / scatter-gather behaviour)
    w24 = {"24h": WINDOWS["24h"]}
    si = window_stats(s_code, t, r_code, t, s_code, amt, w24)
    f["s_in_cnt_24h"] = si["cnt_24h"]; f["s_in_uniq_24h"] = si["distinct_24h"]
    f["s_in_usd_24h"] = si["sum_24h"]
    ro = window_stats(r_code, t, s_code, t, r_code, amt, w24)
    f["r_out_cnt_24h"] = ro["cnt_24h"]; f["r_out_uniq_24h"] = ro["distinct_24h"]
    f["s_passthrough_ratio_24h"] = np.where(si["sum_24h"] > 0, so["sum_24h"] / np.maximum(si["sum_24h"], 1e-9), 0.0)

    # 4) Pair history: first time paying this counterparty? did money flow back before?
    pair = s_code * n_acc + r_code
    rev = r_code * n_acc + s_code
    zeros = np.zeros(n, dtype=np.int64); zf = np.zeros(n)
    ph = window_stats(pair, t, pair, t, zeros, zf, {"all": ALL_HISTORY})
    f["pair_prev_cnt"] = ph["cnt_all"]
    f["pair_first_time"] = (ph["cnt_all"] == 0).astype(np.int8)
    rv = window_stats(rev, t, pair, t, zeros, zf, {"7d": WINDOWS["7d"]})
    f["reverse_flow_7d"] = (rv["cnt_7d"] > 0).astype(np.int8)  # 2-hop cycle: r -> s seen before

    # 5) Account age (hours since first appearance in any role; first appearance <= t)
    first = pd.Series(np.concatenate([t, t])).groupby(codes).min().to_numpy()
    f["s_age_hours"] = (t - first[s_code]) / HOUR_NS
    f["r_age_hours"] = (t - first[r_code]) / HOUR_NS

    # 6) Transaction attributes
    f["log_amount_usd"] = np.log1p(np.clip(amt, 0, None))
    f["round_amount"] = ((df["amount_paid"] % 100 == 0) & (df["amount_paid"] > 0)).astype(np.int8).to_numpy()
    f["is_ach"] = (df["payment_format"].astype(str) == "ACH").astype(np.int8).to_numpy()
    f["cross_bank"] = (df["from_bank"] != df["to_bank"]).astype(np.int8).to_numpy()
    f["self_transfer"] = (s_code == r_code).astype(np.int8)
    f["cross_currency"] = (df["payment_currency"].astype(str) != df["receiving_currency"].astype(str)).astype(np.int8).to_numpy()
    f["hour"] = df["timestamp"].dt.hour.to_numpy(np.int8)
    f["weekday"] = df["timestamp"].dt.day_of_week.to_numpy(np.int8)
    for c in CATEGORICAL:
        f[c] = df[c].astype(str).astype("category")

    # compact dtypes
    for c in f.columns:
        if f[c].dtype == np.float64:
            f[c] = f[c].astype(np.float32)
        elif f[c].dtype == np.int64:
            f[c] = f[c].astype(np.int32)
    return f


def feature_names(f: pd.DataFrame) -> list[str]:
    return list(f.columns)

## 1. Load + attach pattern labels

In [ ]:
t0 = time.time()
raw = pd.read_csv(trans_files[0], dtype={
    "From Bank": "int32", "Account": "string", "To Bank": "int32", "Account.1": "string",
    "Amount Received": "float64", "Receiving Currency": "category", "Amount Paid": "float64",
    "Payment Currency": "category", "Payment Format": "category", "Is Laundering": "int64"})
df = raw.rename(columns={
    "Timestamp": "timestamp", "From Bank": "from_bank", "Account": "from_account",
    "To Bank": "to_bank", "Account.1": "to_account", "Amount Received": "amount_received",
    "Receiving Currency": "receiving_currency", "Amount Paid": "amount_paid",
    "Payment Currency": "payment_currency", "Payment Format": "payment_format", "Is Laundering": "label"})
del raw
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y/%m/%d %H:%M").astype("datetime64[ns]")
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
df["row_id"] = np.arange(len(df), dtype=np.int64)
print(f"loaded {len(df):,} rows in {time.time()-t0:.0f}s; laundering = {int(df['label'].sum()):,}")

# Pattern labels (used only for error analysis in Step 7, never as a feature)
df["pattern"] = pd.Series(pd.NA, index=df.index, dtype="string")
if pattern_files:
    rows, ptype = [], None
    for line in open(pattern_files[0], encoding="utf-8"):
        line = line.strip()
        if line.startswith("BEGIN LAUNDERING ATTEMPT"):
            ptype = line.split("-", 1)[1].strip().split(":")[0].strip()
        elif line.startswith("END LAUNDERING ATTEMPT"):
            ptype = None
        elif line and ptype:
            p = line.split(",")
            rows.append((p[0], int(p[1]), p[2], int(p[3]), p[4], round(float(p[7]), 2), ptype))
    pat = pd.DataFrame(rows, columns=["timestamp", "from_bank", "from_account", "to_bank", "to_account", "amount_paid", "pattern"])
    pat["timestamp"] = pd.to_datetime(pat["timestamp"], format="%Y/%m/%d %H:%M").astype("datetime64[ns]")
    for c in ["from_account", "to_account"]: pat[c] = pat[c].astype("string")
    key = ["timestamp", "from_bank", "from_account", "to_bank", "to_account", "amount_paid"]
    pat = pat.drop_duplicates(key)
    pos = df.loc[df["label"] == 1, key + ["row_id"]].copy(); pos["amount_paid"] = pos["amount_paid"].round(2)
    m = pos.merge(pat, on=key, how="left")
    df.loc[m["row_id"].to_numpy(), "pattern"] = m["pattern"].to_numpy()
df.loc[(df["label"] == 1) & df["pattern"].isna(), "pattern"] = "UNLABELLED"
print(df.loc[df["label"] == 1, "pattern"].value_counts().to_string())

## 2. Time split on the dense period

In [ ]:
day = df["timestamp"].dt.normalize()
daily = df.groupby(day).size()
dense_days = daily.index[daily >= 0.05 * daily.median()]
tail_start = None
sparse = daily.index[daily < 0.05 * daily.median()]
if len(sparse):
    tail_start = sparse.min()
    assert (dense_days < tail_start).all(), "sparse days are not only at the end — check data"

dd = daily.loc[dense_days]
cum = dd.cumsum() / dd.sum()
cut1 = dd.index[int(np.argmin(np.abs(cum.to_numpy() - SPLIT[0])))]
cut2 = dd.index[int(np.argmin(np.abs(cum.to_numpy() - SPLIT[0] - SPLIT[1])))]
split = np.where(day <= cut1, "train", np.where(day <= cut2, "val", "test"))
if tail_start is not None:
    split = np.where(day >= tail_start, "tail", split)
df["split"] = pd.Categorical(split, categories=["train", "val", "test", "tail"])

tab = df.groupby("split", observed=True).agg(rows=("label", "size"), laundering=("label", "sum"),
                                             first=("timestamp", "min"), last=("timestamp", "max"))
tab["rate_%"] = 100 * tab["laundering"] / tab["rows"]
print(tab.to_string())
info["split"] = {k: {"rows": int(r.rows), "laundering": int(r.laundering), "rate_pct": round(float(r["rate_%"]), 4),
                     "first": str(r["first"]), "last": str(r["last"])} for k, r in tab.iterrows()}

## 3. Build features (exchange rates estimated from TRAIN rows only)

In [ ]:
usd_rates = estimate_usd_rates(df[df["split"] == "train"])
missing = sorted(set(df["payment_currency"].astype(str)) - set(usd_rates))
print("USD per unit:", {k: round(v, 4) for k, v in usd_rates.items()})
print("currencies without a rate (treated as 1.0):", missing)
info["usd_rates"] = usd_rates; info["currencies_without_rate"] = missing

t0 = time.time()
feats = build_features(df, usd_rates)
info["feature_build_seconds"] = round(time.time() - t0, 1)
print(f"built {feats.shape[1]} features for {len(feats):,} rows in {time.time()-t0:.0f}s")
feats.describe().T.round(3)

## 4. Spot-check on real data: brute-force recomputation of 300 random test rows

In [ ]:
src = (df["from_bank"].astype(str) + "_" + df["from_account"].astype(str)).to_numpy()
dst = (df["to_bank"].astype(str) + "_" + df["to_account"].astype(str)).to_numpy()
ts = df["timestamp"].to_numpy()
by_src = pd.Series(np.arange(len(df))).groupby(src).indices
by_dst = pd.Series(np.arange(len(df))).groupby(dst).indices
H24, D7 = np.timedelta64(24, "h"), np.timedelta64(7, "D")

rng = np.random.default_rng(SEED)
test_idx = np.flatnonzero(df["split"].to_numpy() == "test")
pos_idx = np.flatnonzero((df["split"].to_numpy() == "test") & (df["label"].to_numpy() == 1))
check = np.concatenate([rng.choice(test_idx, 250, replace=False), rng.choice(pos_idx, min(50, len(pos_idx)), replace=False)])
bad = []
for i in check:
    t = ts[i]
    so = by_src[src[i]]; so = so[(ts[so] < t)]
    ri = by_dst[dst[i]]; ri = ri[(ts[ri] < t)]
    exp = {
        "s_out_cnt_24h": int((ts[so] >= t - H24).sum()),
        "s_out_uniq_24h": len(set(dst[so[ts[so] >= t - H24]])),
        "r_in_uniq_7d": len(set(src[ri[ts[ri] >= t - D7]])),
        "pair_prev_cnt": int((dst[so] == dst[i]).sum()),
    }
    for k, v in exp.items():
        if feats.at[i, k] != v:
            bad.append((int(i), k, float(feats.at[i, k]), v))
print(f"checked {len(check)} rows x 4 features; mismatches: {len(bad)}")
print(bad[:10])
info["spot_check"] = {"rows": int(len(check)), "features": 4, "mismatches": len(bad)}
assert not bad, "feature mismatch — do not use these features"

## 5. Export for the laptop

In [ ]:
meta = df[["row_id", "timestamp", "split", "label", "pattern"]].reset_index(drop=True)
full = pd.concat([meta, feats.reset_index(drop=True)], axis=1)
full["label"] = full["label"].astype("int8")

def sample(part):
    rng = np.random.default_rng(SEED)
    keep = (part["label"] == 1) | (rng.random(len(part)) < NEGATIVE_RATE)
    out = part[keep].copy()
    out["weight"] = np.where(out["label"] == 1, 1.0, 1.0 / NEGATIVE_RATE).astype("float32")
    return out

files = {
    "train_sample.parquet": sample(full[full["split"] == "train"]),
    "val_sample.parquet": sample(full[full["split"] == "val"]),
    "val_full.parquet": full[full["split"] == "val"],
    "test_full.parquet": full[full["split"] == "test"],
    "tail.parquet": full[full["split"] == "tail"],
}
info["files"] = {}
for name, part in files.items():
    part.to_parquet(OUT / name, index=False)
    mb = (OUT / name).stat().st_size / 1e6
    info["files"][name] = {"rows": int(len(part)), "laundering": int(part["label"].sum()), "MB": round(mb, 1)}
    print(f"{name:22s} rows={len(part):>9,}  laundering={int(part['label'].sum()):>5,}  {mb:6.1f} MB")

info["negative_rate"] = NEGATIVE_RATE; info["seed"] = SEED
info["features"] = list(feats.columns)
(OUT / "feature_list.json").write_text(json.dumps(list(feats.columns), indent=1))
(OUT / "split_info.json").write_text(json.dumps(info, indent=1, default=str))
print("total MB:", round(sum(v["MB"] for v in info["files"].values()), 1))